# Sparkle Movie — Exploration & Analyse des données

**Contexte :** Nous travaillons pour une plateforme de streaming vidéo qui souhaite améliorer l'expérience utilisateur en proposant des recommandations personnalisées. Nous utilisons le dataset **MovieLens** pour analyser les comportements des utilisateurs et préparer un système de recommandation.

**Problématique :** Comment recommander des films pertinents à un utilisateur en se basant sur ses préférences passées et celles d'utilisateurs similaires ?

---

## 1. Préparation de l'environnement

### 1.1 Installation des dépendances

Les librairies nécessaires pour ce projet :

In [16]:
# Installation des librairies (à exécuter une seule fois)
# !pip install pyspark matplotlib seaborn pandas

### 1.2 Configuration de la session Spark

In [17]:
import os

# Configuration Java (nécessaire sur Windows)
os.environ["JAVA_HOME"] = "C:/Program Files/Microsoft/jdk-21.0.10.7-hotspot"

from pyspark.sql import SparkSession

# Création de la session Spark
spark = SparkSession.builder \
    .appName("SparkleMovie") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Réduction du niveau de log pour plus de lisibilité
spark.sparkContext.setLogLevel("WARN")

print(f"Session Spark créée avec succès !")
print(f"Version Spark : {spark.version}")
print(f"Application ID : {spark.sparkContext.applicationId}")

Session Spark créée avec succès !
Version Spark : 4.1.1
Application ID : local-1773752875883


### 1.3 Vérification de l'environnement

On s'assure que toutes les librairies sont disponibles et que l'environnement est prêt.

In [18]:
import pyspark
import pandas as pd
import matplotlib
import seaborn

print("Versions des librairies :")
print(f"  PySpark   : {pyspark.__version__}")
print(f"  Pandas    : {pd.__version__}")
print(f"  Matplotlib: {matplotlib.__version__}")
print(f"  Seaborn   : {seaborn.__version__}")
print()
print("Environnement prêt !")

Versions des librairies :
  PySpark   : 4.1.1
  Pandas    : 3.0.1
  Matplotlib: 3.10.8
  Seaborn   : 0.13.2

Environnement prêt !


## 2. Chargement et exploration des données

### 2.1 Importation de csv dans des DataFrames Spark

In [19]:
df_ratings = spark.read.csv("ml-32m/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv("ml-32m/movies.csv", header=True, inferSchema=True)
# header=True : utilise la première ligne pour les noms de colonnes
# inferSchema=True : détecte automatiquement si une colonne est un nombre ou du texte

In [20]:
print("Aperçu des notes :")
df_ratings.show(10)

Aperçu des notes :
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
|     1|     30|   5.0|944249077|
|     1|     32|   5.0|943228858|
|     1|     34|   2.0|943228491|
|     1|     36|   1.0|944249008|
|     1|     80|   5.0|944248943|
|     1|    110|   3.0|943231119|
|     1|    111|   5.0|944249008|
+------+-------+------+---------+
only showing top 10 rows


In [21]:
print("Aperçu des films :")
df_movies.show(10)

Aperçu des films :
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
+-------+--------------------+--------------------+
only showing top 10 rows


### 2.2 Nettoyage des données

In [22]:
print(f"Total ratings: {df_ratings.count()}")
print(f"Total movies: {df_movies.count()}")

Total ratings: 32000204
Total movies: 87585


In [23]:
df_ratings_final = df_ratings.na.drop(subset=["userId", "movieId", "rating"]) \
                             .dropDuplicates(["userId", "movieId"]) \
                             .filter("rating >= 0 AND rating <= 5")
#supprime les lignes vides, les doublons et les notes inferieures à 0 et supérieur à 5

In [24]:
df_movies_final = df_movies.na.drop(subset=["movieId", "title"]) \
                           .dropDuplicates(["movieId"])
#supprime les lignes vides et les doublons

In [25]:
print(f"Lignes après nettoyage : {df_ratings_final.count()}")
print(f"Lignes après nettoyage : {df_movies_final.count()}")

Lignes après nettoyage : 32000204
Lignes après nettoyage : 87585


### 2.3 Analyse des données

In [26]:
df_combined = df_ratings_final.join(df_movies_final, on="movieId", how="inner")
# jointure des dataset

In [27]:
from pyspark.sql import functions as F

In [ ]:
top_rated_movies = df_combined.groupBy("title") \
    .agg(
        F.avg("rating").alias("moyenne_note"),
        F.count("rating").alias("nombre_de_votes")
    ) \
    .filter("nombre_de_votes > 100") \
    .orderBy(F.desc("moyenne_note"))

top_rated_movies.show(10, truncate=False)
# Groupement par titre, calcul de la moyenne et du nombre de votes

+--------------------------------+------------------+---------------+
|title                           |moyenne_note      |nombre_de_votes|
+--------------------------------+------------------+---------------+
|Planet Earth II (2016)          |4.4468302658486705|1956           |
|Planet Earth (2006)             |4.444369063772049 |2948           |
|Band of Brothers (2001)         |4.426538598363572 |2811           |
|Shawshank Redemption, The (1994)|4.404613860039444 |102929         |
|Cosmos                          |4.330081300813008 |615            |
|Godfather, The (1972)           |4.317030403371463 |66440          |
|Parasite (2019)                 |4.312253641816624 |11670          |
|Blue Planet II (2017)           |4.300085984522786 |1163           |
|Twin Peaks (1989)               |4.298684210526316 |1140           |
|Twelve Angry Men (1954)         |4.28619153674833  |449            |
+--------------------------------+------------------+---------------+
only showing top 10 

In [30]:
df_genres = df_combined.withColumn("genre_individuel", F.explode(F.split(F.col("genres"), "\|")))
# Séparation de la chaîne de caractères en liste, puis "exploser" la liste en plusieurs lignes

<>:1: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<>:1: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
C:\Users\aubru\AppData\Local\Temp\ipykernel_32228\3537201875.py:1: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
  df_genres = df_combined.withColumn("genre_individuel", F.explode(F.split(F.col("genres"), "\|")))


In [31]:
genre_popularity = df_genres.groupBy("genre_individuel") \
    .agg(
        F.count("rating").alias("nombre_de_notes"),
        F.avg("rating").alias("note_moyenne")
    ) \
    .orderBy(F.desc("nombre_de_notes"))

genre_popularity.show()
# Calculer la popularité (nombre de notes par genre)

+------------------+---------------+------------------+
|  genre_individuel|nombre_de_notes|      note_moyenne|
+------------------+---------------+------------------+
|             Drama|       13973271|3.6824540581800784|
|            Comedy|       11206925|3.4323858239436777|
|            Action|        9665213| 3.476407141777424|
|          Thriller|        8679464|3.5317020152396505|
|         Adventure|        7590522|3.5234385724723545|
|            Sci-Fi|        5717337|3.4916991949223912|
|           Romance|        5524615|3.5450028644529983|
|             Crime|        5373051|3.6917711184948736|
|           Fantasy|        3702759| 3.512174705402107|
|          Children|        2731841|3.4392409733948646|
|           Mystery|        2615322| 3.673102967818112|
|            Horror|        2492315|3.3071549944529486|
|         Animation|        2214562|3.6153322869262636|
|               War|        1594110|3.7916994435766664|
|              IMAX|        1494179| 3.593312447